# 05 - Causal patching, TF-IDF setting (GPU)

**Settings:** Accelerator **GPU T4 x1** | Internet **On** | Secret `HF_TOKEN` (optional: the ungated `unsloth` mirror of Llama-3.2-3B is used otherwise). No dataset input: the code is cloned from GitHub (branch `cf-tfidf`), the data comes from HF at pinned revisions.

**Runtime:** ~10 min preflight, then the main run stops itself after 10 h (~2-2.5 min per pair; 259 pairs in total). Pairs run in a fixed random order, so a partial run is a random sample. Re-running continues where it stopped.

**What it measures** (`causal_patching.py` v5): a TF-IDF clean question (hop 1 correct) vs its counterfactual (`cf_v1`: only the entry entity swapped, hop 1 fails). Test split only; failure kinds *hedge* + *other_wrong_entity*.
* **A** - copy the clean run's states into the CF run at block L (entity tokens / tokens after the entity / last token only); does hop 1 name the right bridge entity again? Plus a **donor control** (a third entity's states, computed in the same prompt; should not restore) and an all-positions **sanity** check (should restore ~100 %).
* **B** - transplant only the state at `</hop1>`.
* **C** - after each patch, does the **deployed** probe (shared hop12, block 10, calibrated, τ = 0.639) change its verdict?

**Check in the preflight output (STEP 1) before letting the main run go:**
* `minimal pairs ...: 259` and `usable pairs: 2  skipped: {}`
* `entity span length: median ...` small numbers (2-12)
* `HOOK SELF-TEST: ... -> PASS`
* `PROBE-LINK SELF-CHECK ... -> OK` (cosine ~1.0 and stored P = recomputed P; on WARNING, A and B are still valid, only C is doubtful)

Tip: after the preflight passed once, use **Save Version -> Save & Run All (Commit)** so the main run goes in the background. Download `patching_tfidf_results.zip` from the Output tab.

In [ ]:
# ===== Internet check (Settings -> Internet must be On; committed runs use the SAVED setting) =====
import socket
for host in ('github.com', 'huggingface.co'):
    try:
        socket.gethostbyname(host)
    except OSError:
        raise SystemExit(f'No internet ({host} does not resolve). Turn Internet ON in the notebook settings, then run again.')
print('internet OK')

In [ ]:
# ===== Setup: clone the code, HF token, GPU =====
import os, subprocess, sys, time, zipfile
REPO_URL = 'https://github.com/ANLPproject/LinguaFranca.git'
BRANCH = 'cf-tfidf'
WORK = '/kaggle/working/LinguaFranca'
if not os.path.exists(WORK):
    subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, REPO_URL, WORK], check=True)
else:
    subprocess.run(['git', '-C', WORK, 'pull'], check=True)
os.chdir(WORK); sys.path.insert(0, WORK)
print(subprocess.run(['git', 'log', '-1', '--oneline'], capture_output=True, text=True).stdout)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')    # read by the script from the environment
    print('HF_TOKEN loaded from Kaggle secrets')
except Exception as e:
    print(f'No HF_TOKEN secret ({e.__class__.__name__}); the ungated model mirror will be used')
import torch
assert torch.cuda.is_available(), 'No GPU: Session options -> Accelerator -> GPU T4 x1, then restart.'
print('CUDA:', torch.cuda.device_count(), 'x', torch.cuda.get_device_name(0))

def run(cmd):
    '''Stream a command's output; return its exit code.'''
    print('$', ' '.join(cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    for line in p.stdout:
        if 'it/s]' in line or 's/it]' in line:
            continue
        print(line, end='')
    return p.wait()

OUT = '/kaggle/working/patching_tfidf'
PRE = '/kaggle/working/patching_preflight'

In [ ]:
# ===== STEP 1 - PREFLIGHT (~10 min): 2 pairs, 2 blocks, hook self-test + probe-link self-check. Aborts loudly if wrong. =====
rc = run([sys.executable, 'causal_patching.py', '--out', PRE, '--max-pairs', '2', '--layers', '0,10',
          '--boundary-layers', '10', '--max-new-tokens', '40', '--answer-tokens', '60'])
assert rc == 0 and os.path.exists(f'{PRE}/summary.md'), 'PREFLIGHT FAILED - read the log above. Nothing long was started.'
print('\nPREFLIGHT PASSED')

In [ ]:
# ===== STEP 2 - MAIN RUN (stops itself after 10 h; resumable: re-running continues where it stopped) =====
BUDGET_MIN = 600
t_start = time.time()
for attempt in range(1, 4):
    left = BUDGET_MIN - (time.time() - t_start) / 60
    if left < 10:
        break
    rc = run([sys.executable, 'causal_patching.py', '--out', OUT, '--max-minutes', f'{left:.0f}'])
    if rc == 0:
        break
    print(f'\n[attempt {attempt}] exit code {rc} - resuming (finished pairs are skipped)')
print('main run exit code:', rc)

In [ ]:
# ===== STEP 3 - package (always run this): rebuild the summary from whatever pairs finished =====
run([sys.executable, 'causal_patching.py', '--out', OUT, '--summarize-only'])
zpath = '/kaggle/working/patching_tfidf_results.zip'
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for d in (OUT, PRE):
        for root, _, files in os.walk(d):
            for f in files:
                p = os.path.join(root, f)
                z.write(p, os.path.relpath(p, os.path.dirname(d)))
print(f'{zpath}: {os.path.getsize(zpath) / 1e6:.1f} MB')